In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, desc

In [3]:
PROJECT_ROOT = Path.cwd()

RISK_SIGNALS_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "entity_risk_signals_v2.csv"
RISK_SIGNALS_PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "entity_risk_signals_v2.parquet"
TABLES_DIR = PROJECT_ROOT / "artifacts" / "tables"

print("CSV path:", RISK_SIGNALS_CSV_PATH)
print("Parquet path:", RISK_SIGNALS_PARQUET_PATH)

CSV path: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/processed/entity_risk_signals_v2.csv
Parquet path: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/processed/entity_risk_signals_v2.parquet


In [4]:
risk_signals_df = pd.read_csv(RISK_SIGNALS_CSV_PATH)
risk_signals_df.to_parquet(RISK_SIGNALS_PARQUET_PATH, index=False)

print("Saved parquet to:", RISK_SIGNALS_PARQUET_PATH)

Saved parquet to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/processed/entity_risk_signals_v2.parquet


In [5]:
spark = (
    SparkSession.builder
    .appName("uk-entity-risk-summary")
    .getOrCreate()
)

print("Spark session started.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/21 19:52:00 WARN Utils: Your hostname, External-Brain.local, resolves to a loopback address: 127.0.0.1; using 10.136.109.4 instead (on interface en0)
26/03/21 19:52:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/21 19:52:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session started.


In [6]:
spark_df = spark.read.parquet(str(RISK_SIGNALS_PARQUET_PATH))

print("Schema:")
spark_df.printSchema()

print("Preview:")
spark_df.show(5, truncate=False)

Schema:
root
 |-- entity_id: string (nullable = true)
 |-- entity_name: string (nullable = true)
 |-- post_town: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- company_category: string (nullable = true)
 |-- company_status: string (nullable = true)
 |-- incorporation_date: string (nullable = true)
 |-- account_category: string (nullable = true)
 |-- num_mort_charges: long (nullable = true)
 |-- num_mort_outstanding: long (nullable = true)
 |-- num_mort_part_satisfied: long (nullable = true)
 |-- num_mort_satisfied: long (nullable = true)
 |-- sic_text_1: string (nullable = true)
 |-- new_entity_flag: long (nullable = true)
 |-- missing_location_flag: long (nullable = true)
 |-- no_accounts_filed_flag: long (nullable = true)
 |-- has_outstanding_mortgage_flag: long (nullable = true)
 |-- mixed_mortgage_profile_flag: long (nullable = true)
 |-- review_priority_score: long (nullable = true)
 |-- review_priority_band: string (nu

In [7]:
print("Total rows:", spark_df.count())

print("Distinct values in new_entity_flag:")
spark_df.select("new_entity_flag").distinct().show()

print("Distinct values in missing_location_flag:")
spark_df.select("missing_location_flag").distinct().show()

print("Distinct values in no_accounts_filed_flag:")
spark_df.select("no_accounts_filed_flag").distinct().show()

Total rows: 4191535
Distinct values in new_entity_flag:
+---------------+
|new_entity_flag|
+---------------+
|              0|
|              1|
+---------------+

Distinct values in missing_location_flag:
+---------------------+
|missing_location_flag|
+---------------------+
|                    0|
|                    1|
+---------------------+

Distinct values in no_accounts_filed_flag:
+----------------------+
|no_accounts_filed_flag|
+----------------------+
|                     0|
|                     1|
+----------------------+



In [8]:
priority_band_summary = (
    spark_df.groupBy("review_priority_band")
    .agg(count("*").alias("entity_count"))
    .orderBy(desc("entity_count"))
)

priority_band_summary.show(truncate=False)

+--------------------+------------+
|review_priority_band|entity_count|
+--------------------+------------+
|Low                 |4175779     |
|Medium              |15558       |
|High                |198         |
+--------------------+------------+



In [9]:
signal_trigger_summary = spark_df.select(
    spark_sum(col("new_entity_flag")).alias("new_entity_flag_count"),
    spark_sum(col("missing_location_flag")).alias("missing_location_flag_count"),
    spark_sum(col("no_accounts_filed_flag")).alias("no_accounts_filed_flag_count"),
    spark_sum(col("has_outstanding_mortgage_flag")).alias("has_outstanding_mortgage_flag_count"),
    spark_sum(col("mixed_mortgage_profile_flag")).alias("mixed_mortgage_profile_flag_count")
)

signal_trigger_summary.show(truncate=False)

+---------------------+---------------------------+----------------------------+-----------------------------------+---------------------------------+
|new_entity_flag_count|missing_location_flag_count|no_accounts_filed_flag_count|has_outstanding_mortgage_flag_count|mixed_mortgage_profile_flag_count|
+---------------------+---------------------------+----------------------------+-----------------------------------+---------------------------------+
|718298               |84                         |1137714                     |563923                             |180104                           |
+---------------------+---------------------------+----------------------------+-----------------------------------+---------------------------------+



In [10]:
avg_score_by_band = (
    spark_df.groupBy("review_priority_band")
    .agg(avg("review_priority_score").alias("avg_review_priority_score"))
    .orderBy(desc("avg_review_priority_score"))
)

avg_score_by_band.show(truncate=False)

+--------------------+-------------------------+
|review_priority_band|avg_review_priority_score|
+--------------------+-------------------------+
|High                |4.0                      |
|Medium              |3.0                      |
|Low                 |0.6113007896251215       |
+--------------------+-------------------------+



In [11]:
top_high_sic = (
    spark_df.filter(col("review_priority_band") == "High")
    .groupBy("sic_text_1")
    .agg(count("*").alias("entity_count"))
    .orderBy(desc("entity_count"))
)

top_high_sic.show(20, truncate=False)

+-----------------------------------------------------------------------------------------------+------------+
|sic_text_1                                                                                     |entity_count|
+-----------------------------------------------------------------------------------------------+------------+
|68100 - Buying and selling of own real estate                                                  |98          |
|68209 - Other letting and operating of own or leased real estate                               |49          |
|64209 - Activities of other holding companies n.e.c.                                           |13          |
|41100 - Development of building projects                                                       |11          |
|55100 - Hotels and similar accommodation                                                       |3           |
|59111 - Motion picture production activities                                                   |2           |
|

In [12]:
high_accounts_summary = (
    spark_df.filter(col("review_priority_band") == "High")
    .groupBy("account_category")
    .agg(count("*").alias("entity_count"))
    .orderBy(desc("entity_count"))
)

high_accounts_summary.show(20, truncate=False)

+-----------------+------------+
|account_category |entity_count|
+-----------------+------------+
|NO ACCOUNTS FILED|198         |
+-----------------+------------+



In [13]:
high_mortgage_summary = (
    spark_df.filter(col("review_priority_band") == "High")
    .select(
        avg("num_mort_charges").alias("avg_num_mort_charges"),
        avg("num_mort_outstanding").alias("avg_num_mort_outstanding")
    )
)

high_mortgage_summary.show(truncate=False)

+--------------------+------------------------+
|avg_num_mort_charges|avg_num_mort_outstanding|
+--------------------+------------------------+
|3.5757575757575757  |2.217171717171717       |
+--------------------+------------------------+



In [14]:
band_by_accounts_summary = (
    spark_df.groupBy("review_priority_band", "account_category")
    .agg(count("*").alias("entity_count"))
    .orderBy("review_priority_band", desc("entity_count"))
)

band_by_accounts_summary.show(30, truncate=False)

+--------------------+---------------------------+------------+
|review_priority_band|account_category           |entity_count|
+--------------------+---------------------------+------------+
|High                |NO ACCOUNTS FILED          |198         |
|Low                 |MICRO ENTITY               |1582014     |
|Low                 |TOTAL EXEMPTION FULL       |1145854     |
|Low                 |NO ACCOUNTS FILED          |1121964     |
|Low                 |UNAUDITED ABRIDGED         |145281      |
|Low                 |FULL                       |62309       |
|Low                 |SMALL                      |56491       |
|Low                 |AUDIT EXEMPTION SUBSIDIARY |31243       |
|Low                 |GROUP                      |21240       |
|Low                 |MEDIUM                     |5695        |
|Low                 |TOTAL EXEMPTION SMALL      |1311        |
|Low                 |AUDITED ABRIDGED           |1118        |
|Low                 |ACCOUNTS TYPE NOT 

In [15]:
priority_band_summary_pd = priority_band_summary.toPandas()
signal_trigger_summary_pd = signal_trigger_summary.toPandas()
avg_score_by_band_pd = avg_score_by_band.toPandas()
top_high_sic_pd = top_high_sic.limit(50).toPandas()
high_accounts_summary_pd = high_accounts_summary.limit(50).toPandas()
high_mortgage_summary_pd = high_mortgage_summary.toPandas()
band_by_accounts_summary_pd = band_by_accounts_summary.limit(100).toPandas()

In [16]:
priority_band_summary_pd.to_csv(TABLES_DIR / "priority_band_summary.csv", index=False)
signal_trigger_summary_pd.to_csv(TABLES_DIR / "signal_trigger_summary.csv", index=False)
avg_score_by_band_pd.to_csv(TABLES_DIR / "avg_score_by_band.csv", index=False)
top_high_sic_pd.to_csv(TABLES_DIR / "top_high_sic_summary.csv", index=False)
high_accounts_summary_pd.to_csv(TABLES_DIR / "high_accounts_summary.csv", index=False)
high_mortgage_summary_pd.to_csv(TABLES_DIR / "high_mortgage_summary.csv", index=False)
band_by_accounts_summary_pd.to_csv(TABLES_DIR / "band_by_accounts_summary.csv", index=False)

print("Spark summary tables saved to:", TABLES_DIR)

Spark summary tables saved to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/tables


In [17]:
print("Priority band summary")
display(priority_band_summary_pd)

print("Signal trigger summary")
display(signal_trigger_summary_pd)

print("Top SIC among High-priority entities")
display(top_high_sic_pd.head(20))

Priority band summary


,review_priority_band,entity_count
0,Low,4175779
1,Medium,15558
2,High,198


Signal trigger summary


,new_entity_flag_count,missing_location_flag_count,no_accounts_filed_flag_count,has_outstanding_mortgage_flag_count,mixed_mortgage_profile_flag_count
0,718298,84,1137714,563923,180104


Top SIC among High-priority entities


,sic_text_1,entity_count
0,68100 - Buying and selling of own real estate,98
1,68209 - Other letting and operating of own or ...,49
2,64209 - Activities of other holding companies ...,13
3,41100 - Development of building projects,11
4,55100 - Hotels and similar accommodation,3
5,59111 - Motion picture production activities,2
6,64999 - Financial intermediation not elsewhere...,2
7,74990 - Non-trading company,2
8,70100 - Activities of head offices,2
9,78200 - Temporary employment agency activities,1


In [18]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
